<a href="https://colab.research.google.com/github/carolshayle/Python/blob/main/Building_Overlaps_Removal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install geopandas shapely
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from google.colab import files
import os

In [2]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Load data
gdf = gpd.read_file(file_name)

# Basic cleanup: Remove invalid geometries and ensure everything is a polygon
gdf = gdf[gdf.geometry.notnull()]
gdf.geometry = gdf.geometry.buffer(0) # Fixes self-intersections

# Sort by confidence score if it exists (highly recommended for Open Buildings)
if 'confidence' in gdf.columns:
    gdf = gdf.sort_values(by='confidence', ascending=False).reset_index(drop=True)

print(f"Loaded {len(gdf)} buildings.")

Saving Balcad_Buildings.zip to Balcad_Buildings.zip
Loaded 2538 buildings.


In [3]:
def resolve_overlaps(gdf):
    # Working on a copy
    clean_gdf = gdf.copy()

    for i in range(len(clean_gdf)):
        # Get the geometry of the current building (the "cutter")
        cutter = clean_gdf.iloc[i].geometry

        # Look at all buildings after this one in the list
        for j in range(i + 1, len(clean_gdf)):
            target = clean_gdf.iloc[j].geometry

            # If they overlap, subtract the cutter from the target
            if cutter.intersects(target):
                new_geom = target.difference(cutter)

                # Update the geometry in the dataframe
                clean_gdf.at[j, 'geometry'] = new_geom

    # Remove buildings that were completely overlapped (now empty)
    clean_gdf = clean_gdf[~clean_gdf.geometry.is_empty]

    # Optional: Convert MultiPolygons to Polygons if clipping created separate parts
    # (Only keeps the largest part to avoid tiny slivers)
    clean_gdf.geometry = clean_gdf.geometry.apply(lambda x: max(x.geoms, key=lambda p: p.area) if isinstance(x, MultiPolygon) else x)

    return clean_gdf

# Run the function
print("Cleaning overlaps...")
cleaned_gdf = resolve_overlaps(gdf)
print(f"Done. {len(cleaned_gdf)} buildings remaining.")

Cleaning overlaps...
Done. 2532 buildings remaining.


In [4]:
output_name = "cleaned_buildings_somalia.geojson"
cleaned_gdf.to_file(output_name, driver='GeoJSON')
files.download(output_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>